# 0 · Validação — reproduzindo a decomposição do BNDES

**Objetivo:** verificar, com fontes independentes, se o encadeamento de extração
reproduz o Estudo do BNDES nº 67/2026 (*"Atualizando a decomposição do pagamento
de juros nominais, 2002–2025"* — Gráficos 1 e 2).

**Fontes usadas aqui:**
- Juros nominais da dívida federal: **BCB/SGS série 4616** ("Juros nominais – fluxo mensal, R$ milhões", estatísticas fiscais);
- Swap cambial (resultado caixa): **BCB/SGS série 29522**;
- PIB nominal: **BCB/SGS série 4380** (soma dos 12 meses).

**O que o BNDES reporta:** em 2024 os swaps **aumentaram** o dispêndio em juros em
~1 p.p. do PIB (juros com swaps 8,1% do PIB; ex-swaps 7,1%); em 2025 **reduziram**
em ~0,8 p.p. (7,9% com swaps; 8,7% ex-swaps).


In [1]:
import sys
from pathlib import Path
p = Path.cwd()
while not (p / "src").exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

from src.viz.graficos import (
    estilo, salvar, AZUL, VERMELHO, VERDE, LARANJA, ROXO, CINZA, CORES_DETENTOR,
)
from src.processa import indicadores as ind

estilo()
print("diretório de dados:", p / "dados" / "processado")


diretório de dados: /home/vanessa/devel/juros_e_divida/dados/processado


In [2]:
val = ind.validacao_bndes()
print("Cálculo próprio vs Estudo BNDES 67/2026 (todas as taxas em % do PIB):")
print(val.to_string(index=False))

Cálculo próprio vs Estudo BNDES 67/2026 (todas as taxas em % do PIB):
 ano  juros_pib_calc  juros_pib_bndes  ex_swap_calc  ex_swap_bndes  swap_pp_calc
2024            8.07             8.10          7.08           7.10          0.98
2025            7.91             7.90          8.74           8.70         -0.83


**Resultado da validação.** As diferenças são ≤ 0,03 p.p. em todas as células —
os números próprios **reproduzem** os Gráficos 1 e 2 do estudo a partir das
séries brutas do BCB, sem consultar o texto do estudo:

- 2024: swaps **contribuíram com +0,98 p.p.** do PIB para o dispêndio com juros
  (prejuízo do BCB no resultado caixa dos swaps);
- 2025: swaps **reduziram −0,83 p.p.** (ganho do BCB no resultado caixa).

Ou seja: quase todo o movimento do dispêndio com juros, em % do PIB, entre
2024 e 2025 foi **resultado do jogo cambial dos swaps** (juros ex-swaps:
7,1% → 8,7% do PIB), não de um custo real da dívida em queda.

In [3]:
j = ind.juros_anuais()
cols = ["ano", "juros_mi", "swap_resultado_mi", "swap_pp_pib",
        "juros_pib_pct", "juros_ex_swap_pib_pct"]
print("Juros nominais anuais (R$ mi; % do PIB) com decomposição do swap — desde 2015:")
print(j[j.ano >= 2015][cols].round(2).to_string(index=False))

Juros nominais anuais (R$ mi; % do PIB) com decomposição do swap — desde 2015:
 ano     juros_mi  swap_resultado_mi  swap_pp_pib  juros_pib_pct  juros_ex_swap_pib_pct
2015   501,785.90         -89,657.13         1.50           8.37                   6.87
2016   407,024.29          75,562.38        -1.21           6.49                   7.70
2017   400,825.78           7,032.55        -0.11           6.09                   6.19
2018   379,183.90         -15,125.03         0.22           5.41                   5.20
2019   367,281.71          -7,639.55         0.10           4.97                   4.87
2020   312,426.74         -40,800.81         0.54           4.11                   3.57
2021   448,391.19         -22,323.93         0.25           4.98                   4.73
2022   586,426.69          92,265.40        -0.92           5.82                   6.73
2023   718,293.71          70,871.65        -0.65           6.56                   7.21
2024   950,422.84        -115,930.25     

In [4]:
fig, ax = plt.subplots(figsize=(11, 5))
b = j[(j.ano >= 2008) & (j.ano < 2026)]
ax.bar(b.ano - 0.21, b.juros_ex_swap_pib_pct, width=0.4, label="juros ex-swaps", color=AZUL)
ax.bar(b.ano + 0.21, b.juros_pib_pct, width=0.4, label="juros (incl. swaps)", color=CINZA)
ax.axvline(2024.5, color="k", lw=1, ls="--", alpha=0.5)
ax.set_ylim(0, 9.8)
ax.annotate("2024: prejuízo BCB\n(+0,98 p.p. PIB)", xy=(2024, 9.25), fontsize=9, ha="center")
ax.annotate("2025: ganho BCB\n(−0,83 p.p. PIB)", xy=(2025.45, 6.0), fontsize=9, ha="center")
ax.set_ylabel("% do PIB")
ax.set_xlabel("ano")
ax.set_title("Dispêndio com juros nominais da dívida federal: efeito dos swaps (2008–2025)")
ax.legend()
salvar(fig, "00_juros_pib_swaps")

fig -> viz/figs/00_juros_pib_swaps.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/00_juros_pib_swaps.png')